# PaperMind — Notebook 1: Basic RAG

This is the foundation notebook for the PaperMind project. It establishes the canonical pattern we'll reuse across later notebooks:

1. Load secrets from `../.env`
2. Configure Gemini 1.5 Flash as the LLM
3. Configure HuggingFace BGE embeddings (local, free, no API quota)
4. Wire both into LlamaIndex `Settings` (global defaults)
5. Download a sample arXiv paper into `../data/`
6. Index it into `../indexes/<name>/` so subsequent runs reload instead of rebuild
7. Query the index and inspect retrieved source chunks

Path conventions used everywhere in this project:
- `../data/` — raw uploaded papers
- `../indexes/<paper_name>/` — persisted LlamaIndex storage for that paper

## 1. Load environment variables

Reads `GEMINI_API_KEY` from `../.env` (one level above `notebooks/`, alongside `requirements.txt`).

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio

# Jupyter already runs an asyncio event loop. GoogleGenAI calls asyncio.run()
# internally, which fails inside a running loop — nest_asyncio patches asyncio
# to allow nested event loops.
nest_asyncio.apply()

load_dotenv("../.env")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found in ../.env — copy .env.example to .env and fill it in"
print("GEMINI_API_KEY loaded")

GEMINI_API_KEY loaded


## 2. Configure the LLM — Gemini 2.5 Flash

Flash is fast and cheap, which is what we want for RAG response synthesis.

We use `llama-index-llms-google-genai`, which wraps Google's current unified `google.genai` SDK. The older `llama-index-llms-gemini` package is deprecated — it relied on `google.generativeai`, which Google retired in late 2025 and which can't reach current models.

In this new SDK, model names do **not** need the `models/` prefix.

In [2]:
from llama_index.llms.google_genai import GoogleGenAI

llm = GoogleGenAI(model="gemini-2.5-flash", api_key=GEMINI_API_KEY)

## 3. Configure embeddings — BGE base (HuggingFace)

`BAAI/bge-base-en-v1.5` is a strong open-source embedding model. It runs locally on CPU/GPU — no API calls, no rate limits, and no cost. The first run will download the weights (~440 MB) into the HuggingFace cache.

In [3]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 4. Wire models into global Settings

LlamaIndex reads `Settings.llm` and `Settings.embed_model` whenever you build an index or query engine without specifying them explicitly. Setting them once here means every later step picks them up automatically.

In [4]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model

## 5. Download a sample paper

Grabs the *Attention Is All You Need* PDF (Vaswani et al., 2017) from arXiv and saves it to `../data/sample_paper.pdf`. The check-then-download pattern keeps re-runs idempotent.

In [5]:
import requests

DATA_DIR = Path("../data")
INDEX_DIR = Path("../indexes/sample_paper")
DATA_DIR.mkdir(parents=True, exist_ok=True)

pdf_path = DATA_DIR / "sample_paper.pdf"

if pdf_path.exists():
    print(f"Already downloaded: {pdf_path} ({pdf_path.stat().st_size / 1024:.1f} KB)")
else:
    url = "https://arxiv.org/pdf/1706.03762"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    pdf_path.write_bytes(response.content)
    print(f"Downloaded {pdf_path} ({pdf_path.stat().st_size / 1024:.1f} KB)")

Already downloaded: ../data/sample_paper.pdf (2163.3 KB)


## 6. Load the PDF with PyMuPDF

`SimpleDirectoryReader`'s default PDF extractor (`pypdf`) often produces garbage for arXiv papers — multi-column LaTeX layouts and complex object structure trip it up, leaving you with raw PDF metadata instead of text. **PyMuPDF (`fitz`)** is a much stronger extractor and is the right default for academic PDFs.

We open the PDF, pull plain text from each page, and wrap each page as a LlamaIndex `Document` with simple metadata (page number + source filename) so retrieval results stay traceable.

In [6]:
import fitz  # pymupdf
from llama_index.core import Document

pdf = fitz.open(str(pdf_path))
documents = [
    Document(
        text=page.get_text(),
        metadata={"page": i + 1, "source": pdf_path.name},
    )
    for i, page in enumerate(pdf)
]
pdf.close()

print(f"Loaded {len(documents)} pages")
print(f"\nPreview of page 1:\n{documents[0].text[:400]}...")

Loaded 15 pages

Preview of page 1:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
G...


## 7. Build (or load) the vector index

**This is the reusable pattern** — every later notebook will follow this shape:

- If the index dir already has files, reload from disk (cheap).
- Otherwise, embed the documents, build a `VectorStoreIndex`, and persist it.

The persisted form is just a folder of JSON files under `../indexes/<name>/`, so it's portable and easy to inspect.

In [7]:
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage

INDEX_DIR.mkdir(parents=True, exist_ok=True)

if any(INDEX_DIR.iterdir()):
    storage_context = StorageContext.from_defaults(persist_dir=str(INDEX_DIR))
    index = load_index_from_storage(storage_context)
    print(f"Loaded existing index from {INDEX_DIR}")
else:
    index = VectorStoreIndex.from_documents(documents)
    index.storage_context.persist(persist_dir=str(INDEX_DIR))
    print(f"Built new index and persisted to {INDEX_DIR}")

Loaded existing index from ../indexes/sample_paper


## 8. Build a query engine

`similarity_top_k=3` means we retrieve the 3 most relevant chunks per question and feed them to Gemini for synthesis. Three is a sensible default for short papers — large enough to catch context, small enough to stay within Flash's prompt budget.

In [8]:
query_engine = index.as_query_engine(similarity_top_k=3)

## 9. Run test queries

Three questions targeted at the *Attention Is All You Need* paper. For each, we print the synthesised answer plus a snippet of every source chunk used, so we can sanity-check that retrieval actually pulled relevant passages.

In [9]:
queries = [
    "What is the Transformer architecture and how does it differ from RNN- or CNN-based sequence models?",
    "How does multi-head self-attention work, and why use multiple heads instead of one?",
    "Why is positional encoding necessary, and how is it computed in this paper?",
]

for q in queries:
    print("=" * 100)
    print(f"Q: {q}\n")
    response = query_engine.query(q)
    print(f"A: {response}\n")
    print("--- Source nodes ---")
    for i, node in enumerate(response.source_nodes, 1):
        snippet = node.node.get_content()[:300].replace("\n", " ").strip()
        score = f"{node.score:.3f}" if node.score is not None else "n/a"
        print(f"\n[{i}] score={score}")
        print(f"    {snippet}...")
    print()

Q: What is the Transformer architecture and how does it differ from RNN- or CNN-based sequence models?

A: The Transformer is a novel network architecture that completely foregoes recurrence and convolutions, relying instead entirely on attention mechanisms. It follows an encoder-decoder structure. The encoder maps an input sequence into a continuous representation, and the decoder then generates an output sequence one element at a time, operating auto-regressively by consuming previously generated symbols. Both the encoder and decoder are built from stacks of self-attention and point-wise, fully connected layers.

The encoder consists of six identical layers, each with two sub-layers: a multi-head self-attention mechanism and a position-wise fully connected feed-forward network. Residual connections and layer normalization are applied around each sub-layer, with all sub-layers producing outputs of dimension 512.

The decoder also has six identical layers. In addition to the two sub-la